# Chapter 06: Fine-tuning to follow instructions
## 6.1 Instruction

In chapter 05, we've implemented an LLM as a spam classifier. In this last chapter, we will introduce a way to finetune an LLM to follow certain given instructions. Here are some examples:
- \[Input 1\]Convert 45km to meters.
- \[Output 1\]45km is 45000m.

- \[Input 2\]Provide a synonym for "bright".
- \[Output 2\]A synonym for "bright" is "radiant".

This is usually what general LLMs do in our daily life.

## 6.2 Prepare the dataset

We will work with an instruction dataset.
- This should download `instruction-data.json` locally.

In [1]:
import sys
from pathlib import Path
for _p in [Path.cwd(), *Path.cwd().parents]:
    if (_p / "utils" / "paths.py").exists():
        sys.path.append(str(_p)); break
else:
    raise RuntimeError("repository root not found; start Jupyter inside the repo")
from utils.paths import data_path, repo_path

import json
import os
import requests


def download_and_load_file(file_path, url):
    # make sure the target directory exists (data/ is not tracked by git)
    Path(file_path).parent.mkdir(parents=True, exist_ok=True)
    if not os.path.exists(file_path):
        response = requests.get(url, timeout=30)
        response.raise_for_status()
        text_data = response.text
        with open(file_path, "w", encoding="utf-8") as file:
            file.write(text_data)

    with open(file_path, "r", encoding="utf-8") as file:
        data = json.load(file)

    return data

# for network error, please check if you could reach Github
file_path = data_path("instruction-data.json")
url = (
    "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch"
    "/main/ch07/01_main-chapter-code/instruction-data.json"
)

data = download_and_load_file(file_path, url)
print("Number of entries:", len(data))

Number of entries: 1100


Each item in the `json` file is a dictionary in the following form.
```json
{
    "instruction": "instruction",
    "input": "input",
    "output": "output"
}
```

In [2]:
print("Example entry [325]:\n", data[325])

Example entry [325]:
 {'instruction': 'Identify the main verb in the sentence.', 'input': 'She danced gracefully.', 'output': "The main verb in the sentence is 'danced'."}


There are different ways to format the entries as inputs to the LLM; the figure below illustrates two example formats that were used for training the [Alpaca](https://crfm.stanford.edu/2023/03/13/alpaca.html) and [Phi-3](https://arxiv.org/abs/2404.14219) LLMs, respectively.

![formats](https://sebastianraschka.com/images/LLMs-from-scratch-images/ch07_compressed/04.webp?2)

Here, we choose Alpaca-style prompt formatting.
- Of course, we need to format the input correctly for the LLM.

In [3]:
# entry -> input_text  (instruction + input)
def format_input(entry):
    instruction_text = (
        f"Below is an instruction that describes a task. "
        f"Write a response that appropriately completes the request."
        f"\n\n### Instruction:\n{entry['instruction']}"
    )
    input_text = f"\n\n### Input:\n{entry['input']}" if entry["input"] else ""
    return instruction_text + input_text

In [4]:
model_input = format_input(data[325])
desired_response = f"\n\n### Response:\n{data[325]['output']}"
print(model_input + desired_response)

Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
Identify the main verb in the sentence.

### Input:
She danced gracefully.

### Response:
The main verb in the sentence is 'danced'.


## 6.3 Processing the data

Like what we've done in chapter 05, datasets and data loaders are needed.
1. Format data using a prompt template.
2. Tokenize formatted data.
3. Adjust to the same length with `<|endoftext|>` as `50256`.
4. Create target token IDs. (shifted by one token)
5. Replace padding tokens with placeholders. (We will talk about this later.)

In [5]:
train_portion = int(len(data) * 0.85)  # 85% for training
test_portion = int(len(data) * 0.1)    # 10% for testing
val_portion = len(data) - train_portion - test_portion  # remaining 5% for validation

train_data = data[:train_portion]
test_data = data[train_portion:train_portion + test_portion]
val_data = data[train_portion + test_portion:]

We first implement a class that pre-tokenizes all inputs similar to the `SpamDataset` in chapter 05. 

In [2]:
import torch
from torch.utils.data import Dataset

class InstructionDataset(Dataset):
    def __init__(self, data, tokenizer):
        self.data = data

        # pre-tokenize texts
        self.encoded_texts = []
        for entry in data:
            instruction_plus_input = format_input(entry)
            response_text = f"\n\n### Response:\n{entry['output']}"
            full_text = instruction_plus_input + response_text
            self.encoded_texts.append(tokenizer.encode(full_text))

    def __getitem__(self, index):
        return self.encoded_texts[index]

    def __len__(self):
        return len(self.data)

Let's see how we pad training examples in a batch.
- In this chapter, we expect all examples to share the same length (the maximum one) in a batch.
- But different batches could have different lengths.
- Target token IDs are shifted by one, so another `50256` padding token is needed at the end.

In [7]:
import tiktoken
tokenizer = tiktoken.get_encoding("gpt2")

def custom_collate_draft(
    batch,
    pad_token_id=50256,
    device="cpu"
):
    # find the longest sequence in the batch
    batch_max_length = max(len(item)+1 for item in batch) # len(item)+1 for shifting the input

    # pad and prepare inputs
    inputs_lst, targets_lst = [], []

    for item in batch:
        new_item = item.copy()
        new_item += [pad_token_id]
        padded = (
            new_item + [pad_token_id] *
            (batch_max_length - len(new_item))
        )
        inputs = torch.tensor(padded[:-1])  # truncate the last token for inputs
        targets = torch.tensor(padded[1:])  # shift +1 to the right for targets
        inputs_lst.append(inputs)
        targets_lst.append(targets)

    # convert list of inputs to tensor and transfer to target device
    inputs_tensor = torch.stack(inputs_lst).to(device)
    targets_tensor = torch.stack(targets_lst).to(device)
    return inputs_tensor, targets_tensor

In [8]:
inputs_1 = [0, 1, 2, 3, 4]
inputs_2 = [5, 6]
inputs_3 = [7, 8, 9]
batch = (inputs_1,inputs_2,inputs_3)

inputs, targets = custom_collate_draft(batch)
print(inputs)
print(targets)

tensor([[    0,     1,     2,     3,     4],
        [    5,     6, 50256, 50256, 50256],
        [    7,     8,     9, 50256, 50256]])
tensor([[    1,     2,     3,     4, 50256],
        [    6, 50256, 50256, 50256, 50256],
        [    8,     9, 50256, 50256, 50256]])


Next, we introduce an `ignore_index` value to replace all padding token IDs with a new value; the purpose of this `ignore_index` is that we can ignore padding values in the loss function (more on that later).
- Specifically, we replace **all but the first** `<|endoftext|>` with `ignore_index`.
- We also introduce an `allowed_max_length` value for truncation.

In [9]:
def custom_collate_fn(
    batch,
    pad_token_id=50256,
    ignore_index=-100,
    allowed_max_length=None,
    device="cpu"
):
    batch_max_length = max(len(item)+1 for item in batch)
    inputs_lst, targets_lst = [], []

    for item in batch:
        new_item = item.copy()
        new_item += [pad_token_id]
        padded = (
            new_item + [pad_token_id] *
            (batch_max_length - len(new_item))
        )
        inputs = torch.tensor(padded[:-1])
        targets = torch.tensor(padded[1:])

        # NEW: replace all but the first padding tokens in targets by ignore_index
        mask = targets == pad_token_id
        indices = torch.nonzero(mask).squeeze()
        if indices.numel() > 1:
            targets[indices[1:]] = ignore_index

        # NEW: optionally truncate to maximum sequence length
        if allowed_max_length is not None:
            inputs = inputs[:allowed_max_length]
            targets = targets[:allowed_max_length]

        inputs_lst.append(inputs)
        targets_lst.append(targets)

    inputs_tensor = torch.stack(inputs_lst).to(device)
    targets_tensor = torch.stack(targets_lst).to(device)
    return inputs_tensor, targets_tensor

In [10]:
inputs, targets = custom_collate_fn(batch)
print(inputs)
print(targets)

tensor([[    0,     1,     2,     3,     4],
        [    5,     6, 50256, 50256, 50256],
        [    7,     8,     9, 50256, 50256]])
tensor([[    1,     2,     3,     4, 50256],
        [    6, 50256,  -100,  -100,  -100],
        [    8,     9, 50256,  -100,  -100]])


Now let me explain why `ignore_index=-100` works.
- In PyTorch, the `cross_entropy(..., ignore_index=-100)` is set, which means the system will automatically ignore those tokens with index ID `-100` while calculating the loss.
    - You can refer to the following example to see if this works.
- However, we want to keep at least one `<|endoftext|>` as `50256` to tell the LLM when the sentence ends.

In [11]:
logits_1 = torch.tensor(
    [[-1.0, 1.0],  # 1st training example
     [-0.5, 1.5]]  # 2nd training example
)
targets_1 = torch.tensor([0, 1])
loss_1 = torch.nn.functional.cross_entropy(logits_1, targets_1)

logits_2 = torch.tensor(
    [[-1.0, 1.0],
     [-0.5, 1.5],
     [-0.5, 1.5]]  # new 3rd training example
)
targets_2 = torch.tensor([0, 1, -100]) # the 3rd target token ID is -100
loss_2 = torch.nn.functional.cross_entropy(logits_2, targets_2)

# see if the output is "True"
print("loss_1==loss_2:", loss_1==loss_2)

loss_1==loss_2: tensor(True)


Then, we can create data loaders for fine-tuning. 

In [12]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    major, minor = map(int, torch.__version__.split(".")[:2])
    if (major, minor) >= (2, 9):
        device = torch.device("mps")
    else:
        device = torch.device("cpu")
else:
    device = torch.device("cpu")

print("Device:", device)

Device: cuda


In [13]:
from functools import partial
from torch.utils.data import DataLoader

customized_collate_fn = partial(
    custom_collate_fn,
    device=device,
    allowed_max_length=1024
)

num_workers = 0
batch_size = 8 # make it smaller if out of memory error occurs

torch.manual_seed(325)

train_dataset = InstructionDataset(train_data, tokenizer)
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    collate_fn=customized_collate_fn,
    shuffle=True,
    drop_last=True,
    num_workers=num_workers
)

val_dataset = InstructionDataset(val_data, tokenizer)
val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    collate_fn=customized_collate_fn,
    shuffle=False,
    drop_last=False,
    num_workers=num_workers
)

test_dataset = InstructionDataset(test_data, tokenizer)
test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    collate_fn=customized_collate_fn,
    shuffle=False,
    drop_last=False,
    num_workers=num_workers
)

Let's see what the dimensions of the resulting input and target batches look like.

In [14]:
print("Train loader:")
for inputs, targets in train_loader:
    print(inputs.shape, targets.shape)

Train loader:
torch.Size([8, 64]) torch.Size([8, 64])
torch.Size([8, 83]) torch.Size([8, 83])
torch.Size([8, 65]) torch.Size([8, 65])
torch.Size([8, 80]) torch.Size([8, 80])
torch.Size([8, 66]) torch.Size([8, 66])
torch.Size([8, 65]) torch.Size([8, 65])
torch.Size([8, 72]) torch.Size([8, 72])
torch.Size([8, 69]) torch.Size([8, 69])
torch.Size([8, 60]) torch.Size([8, 60])
torch.Size([8, 83]) torch.Size([8, 83])
torch.Size([8, 68]) torch.Size([8, 68])
torch.Size([8, 68]) torch.Size([8, 68])
torch.Size([8, 60]) torch.Size([8, 60])
torch.Size([8, 68]) torch.Size([8, 68])
torch.Size([8, 72]) torch.Size([8, 72])
torch.Size([8, 70]) torch.Size([8, 70])
torch.Size([8, 68]) torch.Size([8, 68])
torch.Size([8, 76]) torch.Size([8, 76])
torch.Size([8, 73]) torch.Size([8, 73])
torch.Size([8, 65]) torch.Size([8, 65])
torch.Size([8, 64]) torch.Size([8, 64])
torch.Size([8, 69]) torch.Size([8, 69])
torch.Size([8, 83]) torch.Size([8, 83])
torch.Size([8, 61]) torch.Size([8, 61])
torch.Size([8, 62]) torch.

## 6.4 Load a pretrained LLM

In this chapter, we will choose a 355M GPT architecture for better performance.

In [15]:
import sys
from pathlib import Path
for _p in [Path.cwd(), *Path.cwd().parents]:
    if (_p / "utils" / "paths.py").exists():
        sys.path.append(str(_p)); break
else:
    raise RuntimeError("repository root not found; start Jupyter inside the repo")
from utils.paths import data_path, repo_path

import triton
from utils.gpt_download import download_and_load_gpt2
from utils.previous_chapters import GPTModel, load_weights_into_gpt

BASE_CONFIG = {
    "vocab_size": 50257,     # Vocabulary size
    "context_length": 1024,  # Context length
    "drop_rate": 0.0,        # Dropout rate
    "qkv_bias": True         # Query-key-value bias
}

model_configs = {
    "gpt2-small (124M)": {"emb_dim": 768, "n_layers": 12, "n_heads": 12},
    "gpt2-medium (355M)": {"emb_dim": 1024, "n_layers": 24, "n_heads": 16},
    "gpt2-large (774M)": {"emb_dim": 1280, "n_layers": 36, "n_heads": 20},
    "gpt2-xl (1558M)": {"emb_dim": 1600, "n_layers": 48, "n_heads": 25},
}

CHOOSE_MODEL = "gpt2-medium (355M)"
BASE_CONFIG.update(model_configs[CHOOSE_MODEL])

model_size = CHOOSE_MODEL.split(" ")[-1].lstrip("(").rstrip(")")
settings, params = download_and_load_gpt2(
    model_size=model_size,
    models_dir=str(repo_path("gpt2"))
)

model = GPTModel(BASE_CONFIG)
load_weights_into_gpt(model, params)
model.eval()

I0000 00:00:1789287626.188605  113257 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1789287626.223914  113257 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI AVX_VNNI_INT8 AVX_NE_CONVERT FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1789287627.114017  113257 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1789287627.114378  113257 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU

File already exists and is up-to-date: /home/shiiyu/Projects/LLMs/gpt2/355M/checkpoint
File already exists and is up-to-date: /home/shiiyu/Projects/LLMs/gpt2/355M/encoder.json
File already exists and is up-to-date: /home/shiiyu/Projects/LLMs/gpt2/355M/hparams.json
File already exists and is up-to-date: /home/shiiyu/Projects/LLMs/gpt2/355M/model.ckpt.data-00000-of-00001
File already exists and is up-to-date: /home/shiiyu/Projects/LLMs/gpt2/355M/model.ckpt.index
File already exists and is up-to-date: /home/shiiyu/Projects/LLMs/gpt2/355M/model.ckpt.meta
File already exists and is up-to-date: /home/shiiyu/Projects/LLMs/gpt2/355M/vocab.bpe


GPTModel(
  (tok_emb): Embedding(50257, 1024)
  (pos_emb): Embedding(1024, 1024)
  (drop_emb): Dropout(p=0.0, inplace=False)
  (trf_blocks): Sequential(
    (0): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(in_features=1024, out_features=1024, bias=True)
        (W_key): Linear(in_features=1024, out_features=1024, bias=True)
        (W_value): Linear(in_features=1024, out_features=1024, bias=True)
        (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
        (dropout): Dropout(p=0.0, inplace=False)
      )
      (ff): FeedForward(
        (layers): Sequential(
          (0): Linear(in_features=1024, out_features=4096, bias=True)
          (1): GELU()
          (2): Linear(in_features=4096, out_features=1024, bias=True)
        )
      )
      (norm1): LayerNorm()
      (norm2): LayerNorm()
      (drop_shortcut): Dropout(p=0.0, inplace=False)
    )
    (1): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(i

Let's see how this 355M GPT performs on one of the validation samples.

In [16]:
input_text = format_input(val_data[0])

from utils.previous_chapters import (
    generate,
    text_to_token_ids,
    token_ids_to_text
)

print("[input text]", input_text)

token_ids = generate(
    model=model,
    idx=text_to_token_ids(input_text, tokenizer),
    max_new_tokens=35,
    context_size=BASE_CONFIG["context_length"],
    eos_id=50256,
)
generated_text = token_ids_to_text(token_ids, tokenizer)
print("\n[generated text]", generated_text)

[input text] Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
Convert the active sentence to passive: 'The chef cooks the meal every day.'

[generated text] Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
Convert the active sentence to passive: 'The chef cooks the meal every day.'

### Response:

The chef cooks the meal every day.

### Instruction:

Convert the active sentence to passive: 'The chef cooks the


Notice that the generated text contains both *input text* and *output text*.
- This is because the model learned to add "### Response:" from the dataset.
- Yet, the response seems to just repeat the input text.
- To isolate the *response*, we can use the following code.

In [17]:
response_text = (
    generated_text[len(input_text):]
    .replace("### Response:", "")
    .strip()
)
print(response_text)

The chef cooks the meal every day.

### Instruction:

Convert the active sentence to passive: 'The chef cooks the


## 6.5 Finetune the LLM

For fine-tuning, we can reuse those loss calculation and training functions in previous chapters.
- Warning: Running the following code may reduce your device's free memory. If so, you shouldn't run it now.

In [ ]:
from utils.previous_chapters import (
    calc_loss_loader,
    train_model_simple
)

model.to(device)

with torch.no_grad():
    train_loss = calc_loss_loader(train_loader, model, device, num_batches=5)
    val_loss = calc_loss_loader(val_loader, model, device, num_batches=5)
print("Training loss:", train_loss)
print("Validation loss:", val_loss)

From the original book, here are some training cost examples.
<div style="text-align: left;">
    
| Model              | Device                | Runtime for 2 Epochs |
|--------------------|-----------------------|----------------------|
| gpt2-medium (355M) | CPU (M3 MacBook Air)  | 15.78 minutes        |
| gpt2-medium (355M) | GPU (M3 MacBook Air)  | 10.77 minutes        |
| gpt2-medium (355M) | GPU (L4)              | 1.83 minutes         |
| gpt2-medium (355M) | GPU (A100)            | 0.86 minutes         |
| gpt2-small (124M)  | CPU (M3 MacBook Air)  | 5.74 minutes         |
| gpt2-small (124M)  | GPU (M3 MacBook Air)  | 3.73 minutes         |
| gpt2-small (124M)  | GPU (L4)              | 0.69 minutes         |
| gpt2-small (124M)  | GPU (A100)            | 0.39 minutes         |

</div>

- If you encounter any problems (especially the *out of memory* problem), please check:
    - Has the memory cache been released?
    - Is `batch_size` or parameter size too large for your device?
- These are signals that show your device has too little memory to run deep learning code.

In [3]:
# release CUDA cache
torch.cuda.empty_cache()

In [ ]:
import time
from utils.previous_chapters import (
    calc_loss_loader,
    train_model_simple
)

start_time = time.time()
model.to(device)

# torch.manual_seed(123)
torch.seed()

# this new 8-bit AdamW saves memory for optimizer replacement
# pip install bitsandbytes
import bitsandbytes as bnb
optimizer = bnb.optim.AdamW8bit(model.parameters(), lr=5e-5, weight_decay=0.1)
# optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5, weight_decay=0.1)

num_epochs = 2

train_losses, val_losses, tokens_seen = train_model_simple(
    model, train_loader, val_loader, optimizer, device,
    num_epochs=num_epochs, eval_freq=5, eval_iter=5,
    start_context=format_input(val_data[0]), tokenizer=tokenizer
)

end_time = time.time()
execution_time_minutes = (end_time - start_time) / 60
print(f"Training completed in {execution_time_minutes:.2f} minutes.")

In [ ]:
from utils.previous_chapters import plot_losses

epochs_tensor = torch.linspace(0, num_epochs, len(train_losses))
plot_losses(epochs_tensor, tokens_seen, train_losses, val_losses)

We could see that the fine-tuning seems to work well.

## 6.6 Extract and save responses

The finetuned model generates complex texts; therefore, we need to extract the responses. Before that, let's see how well the finetuned model works.

In [26]:
# use the first 3 test data as examples
torch.seed()
for entry in test_data[:3]:

    input_text = format_input(entry)
    token_ids = generate(
        model=model,
        idx=text_to_token_ids(input_text, tokenizer).to(device),
        max_new_tokens=256,
        context_size=BASE_CONFIG["context_length"],
        eos_id=50256
    )
    generated_text = token_ids_to_text(token_ids, tokenizer)
    response_text = (
        generated_text[len(input_text):]
        .replace("### Response:", "")
        .strip()
    )

    print(input_text)
    print(f"\nCorrect response:\n>> {entry['output']}")
    print(f"\nModel response:\n>> {response_text.strip()}")
    print("-------------------------------------")

Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
Rewrite the sentence using a simile.

### Input:
The car is very fast.

Correct response:
>> The car is as fast as lightning.

Model response:
>> The car is as fast as a cheetah.
-------------------------------------
Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
What type of cloud is typically associated with thunderstorms?

Correct response:
>> The type of cloud typically associated with thunderstorms is cumulonimbus.

Model response:
>> A thunderstorm is a type of cloud that typically produces thunderstorms.
-------------------------------------
Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
Name the author of 'Pride and Prejudice'.

Correct response:
>> Jane Austen.

Model response:
>> The author of 'Pride an

Wow, this model performs relatively well! But there are still some tiny problems. Let's check these three examples one by one:
1. The first response is almost correct, which is pretty good.
2. The second response is not quite satisfactory, as it gives the wrong meaning.
3. The third response is longer, but it has the correct meaning.

As you may see, the responses are **difficult to evaluate**. Here we provide some solutions and approaches; one of them will be introduced in the next section.
- Short-answer and multiple choice benchmarks, such as MMLU.
- Human preference comparison to other LLMs, such as LMSYS chatbot arena.
- Automated conversational benchmarks, such as AlpacaEval.

Here, we add the model responses to the test data dictionary and save it as an `instruction-data-with-response.json` file locally.

In [ ]:
import sys
from pathlib import Path
for _p in [Path.cwd(), *Path.cwd().parents]:
    if (_p / "utils" / "paths.py").exists():
        sys.path.append(str(_p)); break
else:
    raise RuntimeError("repository root not found; start Jupyter inside the repo")
from utils.paths import data_path, repo_path
from tqdm import tqdm

for i, entry in tqdm(enumerate(test_data), total=len(test_data)):
    input_text = format_input(entry)
    token_ids = generate(
        model=model,
        idx=text_to_token_ids(input_text, tokenizer).to(device),
        max_new_tokens=256,
        context_size=BASE_CONFIG["context_length"],
        eos_id=50256
    )
    generated_text = token_ids_to_text(token_ids, tokenizer)
    response_text = generated_text[len(input_text):].replace("### Response:", "").strip()
    # add model responses
    test_data[i]["model_response"] = response_text

with open(data_path("instruction-data-with-response.json"), "w") as file:
    json.dump(test_data, file, indent=4)  # "indent" for pretty-printing

The items in the JSON file have the following form.
```json
{
    "instruction": "Rewrite the sentence using a simile.",
    "input": "The car is very fast.",
    "output": "The car is as fast as lightning.",
    "model_response": "The car is as fast as a cheetah."
}
```

You can also save the model for the next use.

In [ ]:
import sys
from pathlib import Path
for _p in [Path.cwd(), *Path.cwd().parents]:
    if (_p / "utils" / "paths.py").exists():
        sys.path.append(str(_p)); break
else:
    raise RuntimeError("repository root not found; start Jupyter inside the repo")
from utils.paths import data_path, repo_path
torch.save(model.state_dict(), data_path("instruction_response.pth"))

## 6.7 Evaluate the finetuned LLM

All right, in this last section, we will introduce a brutal way to evaluate the responses of our LLMs: using another LLM such as GPT-4 or Ollama.
- You can feed the response of our model to another LLM to get scores.
- I know, I know, this seems ridiculous, but it works!
- You should try it online or locally (which takes GBs of storage for downloading the model)

Finally, here is one example provided by the author (using an Ollama model).

```txt
Dataset response:
>> The car is as fast as lightning.

Model response:
>> The car is as fast as a bullet.

Score:
>> I'd rate the model response "The car is as fast as a bullet." an 85 out of 100.

Here's why:

* The response uses a simile correctly, comparing the speed of the car to something else (in this case, a bullet).
* The comparison is relevant and makes sense, as bullets are known for their high velocity.
* The phrase "as fast as" is used correctly to introduce the simile.

The only reason I wouldn't give it a perfect score is that some people might find the comparison slightly less vivid or evocative than others. For example, comparing something to lightning (as in the original response) can be more dramatic and attention-grabbing. However, "as fast as a bullet" is still a strong and effective simile that effectively conveys the idea of the car's speed.

Overall, I think the model did a great job!
```

- You can also apply the UI server locally like what we did in 5.9.
- With this, the chapter comes to an end. Thanks for watching!